In [1]:
import numpy as np
from scipy.stats import poisson
import sys
sys.path.append('../../pysimARG')
from add_mutation_truncated import truncated_poisson
from clonal_genealogy import ClonalTree
from ClonalOrigin_pair import ARG
from localtree import LocalTree
from add_mutation_truncated import add_mutation_truncated
from homoplasy_index import homoplasy_index

In [2]:
np.random.seed(100)
tree = ClonalTree(10)

rho_site = 0.5
theta_site = 0.3
L = 100
delta = 5
k = 10

In [3]:
arg = ARG(tree, rho_site, L, delta, k)

In [4]:
node_site = add_mutation_truncated(arg, theta_site)

In [5]:
node_site

array([[ True,  True],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [False, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [False, False],
       [ True, False],
       [ True, False],
       [False, False],
       [False, False],
       [ True, False],
       [ True, False],
       [ True,  True],
       [False, False],
       [False, False],
       [ True, False],
       [False,  True],
       [False, False],
       [ True, False],
       [ True,  True],
       [ True, False],
       [ True, False],
       [False, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [ True, False],
       [False, False]])

In [6]:
homoplasy_index(arg, node_site)

TypeError: unhashable type: 'numpy.ndarray'

In [18]:
n_leaf = arg.n
n_site = arg.node_mat.shape[1]
m_vec = np.zeros(n_site, dtype=int)
s_vec = np.zeros(n_site, dtype=int)

list(range(n_site))

[0, 1]

In [19]:
site_loc = 0
# Get local tree at this site
arg_site = LocalTree(arg, site_loc)
node_vec = arg_site.node_index.astype(int)
node_site_vec = node_site[node_vec - 1, site_loc]  # Convert to 0-indexed

# Compute minimum possible changes
leaf_states = node_site_vec[:n_leaf]
if np.any(leaf_states) and not np.all(leaf_states):
    m_vec[site_loc] = 1

# Compute actual changes using Fitch algorithm
s_site = 0
site_dict = {}

# Initialize leaf nodes
site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}

In [20]:
site_dict

{np.int64(1): {1},
 np.int64(2): {1},
 np.int64(3): {1},
 np.int64(4): {1},
 np.int64(5): {1},
 np.int64(6): {1},
 np.int64(7): {1},
 np.int64(8): {1},
 np.int64(9): {1},
 np.int64(10): {1}}

In [21]:
# Process internal nodes
for i in range(n_leaf, len(node_vec)):
    parent_node = node_vec[i]
    node_indices = np.where(arg_site.edge[:, 0] == parent_node)[0]
    if len(node_indices) == 2:
        # Coalescent structure (two children)
        children_node = arg_site.edge[node_indices, 1].astype(int)
        child_1_states = site_dict[children_node[0]]
        child_2_states = site_dict[children_node[1]]

        intersec = child_1_states & child_2_states
        if len(intersec) == 0:
            # Intersection is empty -> mutation occurred
            site_dict[parent_node] = child_1_states | child_2_states
            s_site += 1
        else:
            # Intersection is not empty -> no mutation
            site_dict[parent_node] = intersec

    elif len(node_indices) == 1:
        # Recombination structure (single child)
        children_node = arg_site.edge[node_indices, 1].astype(int)
        site_dict[parent_node] = site_dict[children_node[0]]

s_vec[site_loc] = s_site

In [30]:
edge = np.zeros((9, 3), dtype=int)
edge[:, 0] = np.array([6, 6, 7, 7, 8, 8, 9, 10, 10])
edge[:, 1] = np.array([4, 5, 3, 6, 1, 2, 7, 8, 9])
edge

array([[ 6,  4,  0],
       [ 6,  5,  0],
       [ 7,  3,  0],
       [ 7,  6,  0],
       [ 8,  1,  0],
       [ 8,  2,  0],
       [ 9,  7,  0],
       [10,  8,  0],
       [10,  9,  0]])

In [38]:
node_vec, node_site_vec

(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]),
 array([ 0.,  0.,  1.,  0.,  1., nan, nan, nan, nan, nan]))

In [39]:
node_vec = np.arange(1, 11)
node_site_vec = np.array([0, 0, 1, 0, 1, np.nan, np.nan, np.nan, np.nan, np.nan])
n_leaf = 5

node_vec, node_site_vec

(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]),
 array([ 0.,  0.,  1.,  0.,  1., nan, nan, nan, nan, nan]))

In [40]:
# Compute minimum possible changes
leaf_states = node_site_vec[:n_leaf]
if np.any(leaf_states) and not np.all(leaf_states):
    print(1)
else:
    print(0)

1


In [41]:
leaf_states

array([0., 0., 1., 0., 1.])

In [42]:
# Compute actual changes using Fitch algorithm
s_site = 0
site_dict = {}

# Initialize leaf nodes
site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}

# Process internal nodes
for i in range(n_leaf, len(node_vec)):
    parent_node = node_vec[i]
    node_indices = np.where(edge[:, 0] == parent_node)[0]
    if len(node_indices) == 2:
        # Coalescent structure (two children)
        children_node = edge[node_indices, 1].astype(int)
        child_1_states = site_dict[children_node[0]]
        child_2_states = site_dict[children_node[1]]

        intersec = child_1_states & child_2_states
        if len(intersec) == 0:
            # Intersection is empty -> mutation occurred
            site_dict[parent_node] = child_1_states | child_2_states
            s_site += 1
        else:
            # Intersection is not empty -> no mutation
            site_dict[parent_node] = intersec

    elif len(node_indices) == 1:
        # Recombination structure (single child)
        children_node = edge[node_indices, 1].astype(int)
        site_dict[parent_node] = site_dict[children_node[0]]

s_site

2

In [44]:
site_dict

{np.int64(1): {0},
 np.int64(2): {0},
 np.int64(3): {1},
 np.int64(4): {0},
 np.int64(5): {1},
 np.int64(6): {0, 1},
 np.int64(7): {1},
 np.int64(8): {0},
 np.int64(9): {1},
 np.int64(10): {0, 1}}